In [ ]:
# Custom
import sys
sys.path.insert(0, '../src')
from args import args
import utils
from snippet_creation import SnippetCreation


## Utils and Inference
import csv
from collections import Counter, OrderedDict
import json
import numpy as np
import pandas as pd
import re

import os
from tqdm import tqdm

In [ ]:
class args(args):

    setting = ["batching", "single_variable"][0]

    experimental_setting = ["vanilla", "full_source_code", "source_code_grep_chunk", "paragraph_code"][1]

    batch_method = ["record", "k_hop"][1]

    saving_path = f"../saved_data/inference/{setting}/{experimental_setting}_{batch_method}"

    variable_column = "VarName"
    program_path_column = "corrected_path"

    data_path = f"../data/batched_data/{batch_method}/batched_data_record_2_MoveStmnt_optimized_para_occ.json"

    platform = ["llm_provider_1", "watsonx", "mse", "litellm", "azure"][0]

    complete_data_path = "../data/static_analyzer_Downloaded_Data/all_application_get_all_variable_info_23_06_2026.json"
    # complete_data_path = "../data/static_analyzer_Downloaded_Data/all_application_get_variable_para_info_formatted_23_06_2026.json"

    batch_size = 15

In [ ]:
complete_data_df = pd.read_json(args.complete_data_path).rename(columns={"ProgramID": "ProgID"})

In [ ]:
import sys
sys.path.insert(0, '..')
from prompts import icl_source_code_prompts_batched

prompt = icl_source_code_prompts_batched.icl_source_code_prompts_batched
print(prompt)

In [ ]:
## Check if the saving directories exist or not; If not, then create them
utils.create_save_directories()

In [ ]:
### Check if the experiments saving directories exist or not; If not, then create them
utils.ensure_exp_dirs_exist(args.saving_path)

## Data Preprocessing

In [ ]:
# data_df.columns

In [ ]:
data_df = pd.read_json(args.data_path)

In [ ]:
initial_batch_data = [eval(x)[0] for x in data_df['batch_variables']]

processed_batched_data = []


for i, curr_batch in enumerate(data_df['batch_2_MoveStmnt_optimized']):
    for key, values in curr_batch[0].items():
        if key in initial_batch_data[i].keys():
            curr_batch[0][key] = initial_batch_data[i][key]

    processed_batched_data.append(curr_batch)

data_df['batch_variables'] = processed_batched_data

In [ ]:
data_df = data_df.drop(["batch_2_MoveStmnt_optimized"], axis=1)

## Source Code Context Helpers

In [ ]:
def read_file_binary_cobol(full_path):
    try:
        with open(full_path, "rb") as f:
            content = f.read()
        return {
            "path": full_path,
            "content": content.decode("latin-1")
        }
    except Exception as e:
        return {
            "error": f"Found file at {full_path} but could not read it: {e}"
        }

In [ ]:
def gather_context_for_batch(batch_variables, app_name):
    """
    Build ONE shared source-code context for all variables in a batch.

    Strategy
    --------
    For every unique corrected_path referenced across all variable dicts in
    the batch, read the entire file once and include its full content in the
    context.  No paragraph slicing is performed — the model receives the
    complete source file so it can observe every occurrence of every variable
    across the whole program.

    Parameters
    ----------
    batch_variables : list[dict]
        List of variable dicts from the batch_variables column.  Each dict
        must contain at minimum: VarName and corrected_path (list-valued).
    app_name : str
        Application name (informational; not used for lookups).

    Returns
    -------
    list[str]  – one marked string per unique source file, suitable for
                 merge_context()
    """

    def _to_list(val):
        """Wrap a scalar value in a list; leave lists/tuples as-is."""
        if val is None:
            return []
        return val if isinstance(val, (list, tuple)) else [val]

    # -----------------------------------------------------------------
    # Step 1: Collect the de-duplicated set of file paths referenced by
    #         any variable in the batch, preserving first-seen order.
    # -----------------------------------------------------------------
    seen_paths = OrderedDict()  # file_path -> None  (ordered set)

    for var_dict in batch_variables:
        if not isinstance(var_dict, dict):
            continue
        for fpath in _to_list(var_dict.get(args.program_path_column)):
            if fpath not in seen_paths:
                seen_paths[fpath] = None

    if not seen_paths:
        return []

    # -----------------------------------------------------------------
    # Step 2: For each unique file, read the entire content and wrap it
    #         with start/end markers.
    # -----------------------------------------------------------------
    source_codes = []

    for file_path in seen_paths:
        file_content = read_file_binary_cobol(full_path=file_path)
        if 'error' in file_content:
            continue

        file_name        = os.path.basename(file_path)
        file_content_str = file_content['content']

        marked_content = f"[start of {file_name}]\n{file_content_str}\n[end of {file_name}]"
        source_codes.append(marked_content)

    return source_codes


def merge_context(source_codes):
    return "\n\n".join(source_codes)

## Validating and Loading the Inference Pipeline

In [ ]:
inference = utils.Inference(args)

In [ ]:
model_names = utils.validate_api_endpoints(args, args.platform)

print(f"Models from {args.platform}: {model_names}\n")

model_names = ['llama3_3_70b', 'gpt_oss_120b', 'gpt_oss_20b', 'qwen_3_8B', 'llama4_17b_16e', 'qwen_3_30b_a3b_thinking', 'gemma_4_31B_it']

print(f"Selected Models from {args.platform}: {model_names}")

In [ ]:
pred_folder = args.saving_path + "/"
pred_folder

In [ ]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


def write_json_record(file_path: str, record: dict) -> None:
    """Append a record to a JSON array file."""
    with open(file_path, 'r+') as jsonfile:
        data = json.load(jsonfile)
        data.append(record)
        jsonfile.seek(0)
        json.dump(data, jsonfile, indent=2, cls=NumpyEncoder)
        jsonfile.truncate()

## Inference — K-Hop Batching with Full Source Code Context

For every row in `data_df`:
1. Retrieve the batch of sibling variables (`batch_variables`).
2. Build **one** shared context by collecting all unique `corrected_path` files referenced across the batch and including each file's **entire content** exactly once.
3. Fill the batched prompt template and run inference.

In [ ]:
file_names = []

for model_name in tqdm(model_names, desc="Models", position=0):
    prediction_df = data_df

    file_name = f"{model_name}_variable_DD_{args.experimental_setting}_{args.batch_method}_.json"
    file_name = utils.update_file_name(
        file_name, pred_folder=pred_folder, extension=".json"
    )

    # Checkpointing Logic — set model_name to '' and assign file_name / checkpoint_index manually
    if model_name == '':
        file_name = 'PLACEHOLDER_FILE_NAME.json'
        checkpoint_index = 0
    else:
        checkpoint_index = 0
        fields = list(prediction_df.columns) + ["prompt_used", "token_statistics", "predictions_DD"]

        with open(pred_folder + file_name, 'w') as jsonfile:
            json.dump([], jsonfile)

    print(f"Starting Inference for {model_name} in {file_name}")
    full_path = pred_folder + file_name

    pbar = tqdm(
        total=len(prediction_df),
        desc=f"Inference [{model_name}]",
        position=1,
        leave=False,
        dynamic_ncols=True
    )

    for i, row in prediction_df.iterrows():
        if i < checkpoint_index:
            pbar.update(1)
            continue
        if i == checkpoint_index:
            print(f"starting inference from index {i}")

        row_data = list(row)

        # -----------------------------------------------------------
        # Resolve batch variables and app context
        # -----------------------------------------------------------
        # batch_variables is a list of dicts (from Batch_Creation Record.ipynb)
        # each dict has 'VarName', 'VarID', 'ParaID', 'ProgramName', etc.
        batch_variables = row['batch_variables'][:args.batch_size]          # list[dict]
        app_name        = row['app_name']                 # str
        # Extract plain variable name strings for the prompt
        batch_var_name_strings = [
            d['VarName'] if isinstance(d, dict) else d
            for d in batch_variables
        ]

        # -----------------------------------------------------------
        # Build the shared context once for the whole batch:
        # collect all unique corrected_path files and include each
        # file's entire content (no paragraph slicing)
        # -----------------------------------------------------------
        try:
            source_codes = gather_context_for_batch(
                batch_variables=batch_variables,
                app_name=app_name
            )
            source_code_string = merge_context(source_codes)

        except Exception as e:
            response    = f"error found: {e}"
            token_stats = f"error found: {e}"
            record = dict(zip(fields, row_data + ["", token_stats, response]))
            write_json_record(full_path, record)
            pbar.update(1)
            continue

        # -----------------------------------------------------------
        # Fill the batched prompt template
        # -----------------------------------------------------------
        current_prompt = prompt[:]
        current_prompt = current_prompt.replace("{{SOURCE_CONTEXT}}", source_code_string)
        current_prompt = current_prompt.replace("{{BATCHED_INPUT}}", str(batch_var_name_strings))

        # -----------------------------------------------------------
        # Run inference
        # -----------------------------------------------------------
        try:
            response, raw_output = inference(
                prompt=current_prompt,
                inference_platform=args.platform,
                model_name=model_name,
                return_raw=True
            )

            token_stats = dict(dict(raw_output)['usage'])
            token_stats = {k: token_stats[k] for k in ('completion_tokens', 'prompt_tokens', 'total_tokens')}

        except Exception as e:
            response    = f"error found: {e}"
            token_stats = f"error found: {e}"

        record = dict(zip(fields, row_data + [current_prompt, token_stats, response]))
        write_json_record(full_path, record)
        pbar.update(1)

    #     break

    # break

    pbar.close()
    file_names.append(file_name)